# VocaLip Thai — Notebook 03
## SSL Fine-Tuning 44 Thai Words → Deployment Checkpoint

โหลด encoder จาก Notebook 02 v2 แล้ว fine-tune 44 คำ  
Train=1632, Val=354, Test=259 (เปิด test หลังเลือก best model เท่านั้น)  
Frozen Experiment 0: Acc 26.25%, Macro-F1 0.2327, Top-3 48.26%.

In [ ]:
# CELL 1 — Setup
import json,time,random,gc,hashlib
from pathlib import Path
import numpy as np,pandas as pd,torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from tqdm.auto import tqdm
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT=Path(r"D:\Face_detection\dataset\word_level_v5\video001")
MANIFEST=ROOT/"word_train_manifest_v5_baseline_clean.csv"
SSL_PATH=ROOT/"models_02_ssl_v2"/"02_ssl_v2_encoder_for_finetuning.pt"
OUT=ROOT/"models_03_ssl_finetune"; OUT.mkdir(exist_ok=True)
BEST=OUT/"03_ssl_finetune_best.pt"; DEPLOY=OUT/"03_vocalip_44words_deployment.pt"
HISTORY=OUT/"03_ssl_finetune_history.csv"; RESULT=OUT/"03_final_results.json"
BATCH=8; EPOCHS=40; PATIENCE=8; NUM_CLASSES=44; AMP=DEVICE.type=="cuda"; SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
assert MANIFEST.exists() and SSL_PATH.exists()
print("Device:",DEVICE); print("SSL:",SSL_PATH)

In [ ]:
# CELL 2 — Data / exact Experiment-0 normalization
df=pd.read_csv(MANIFEST); df["model_class_id"]=df["model_class_id"].astype(int)
tr=df[df.split=="train"].reset_index(drop=True); va=df[df.split=="val"].reset_index(drop=True); te=df[df.split=="test"].reset_index(drop=True)
id2word={int(r.model_class_id):str(r.word) for _,r in df[["model_class_id","word"]].drop_duplicates().iterrows()}
word2id={v:k for k,v in id2word.items()}
MEAN=np.array([.485,.456,.406],np.float32); STD=np.array([.229,.224,.225],np.float32)
def norm(x):
    x=x.astype(np.float32)/255.; x=(x-MEAN[None,None,None,:])/STD[None,None,None,:]
    return np.ascontiguousarray(np.transpose(x,(0,3,1,2)),dtype=np.float32)
def aug(x):
    z=x.astype(np.float32)/255.
    if random.random()<.45: z*=random.uniform(.90,1.10)
    if random.random()<.45:
        m=z.mean((1,2),keepdims=True); z=(z-m)*random.uniform(.90,1.10)+m
    if random.random()<.2: z+=np.random.normal(0,random.uniform(.002,.012),z.shape).astype(np.float32)
    return np.clip(z,0,1)*255.
class DS(Dataset):
    def __init__(self,d,a=False): self.d=d; self.a=a
    def __len__(self): return len(self.d)
    def __getitem__(self,i):
        r=self.d.iloc[i]
        with np.load(Path(str(r.mouth_cache_path))) as q: x=q["frames"].copy()
        if self.a:x=aug(x).round().astype(np.uint8)
        return torch.from_numpy(norm(x)),int(r.model_class_id)
train_loader=DataLoader(DS(tr,True),BATCH,shuffle=True,pin_memory=True)
val_loader=DataLoader(DS(va),BATCH,shuffle=False,pin_memory=True)
test_loader=DataLoader(DS(te),BATCH,shuffle=False,pin_memory=True)
print(len(df),len(tr),len(va),len(te),len(id2word))
assert (len(df),len(tr),len(va),len(te),len(id2word))==(2245,1632,354,259,44)

In [ ]:
# CELL 3 — SSL-compatible model
class FrameCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(
          nn.Conv2d(3,32,3,padding=1,bias=False),nn.BatchNorm2d(32),nn.ReLU(True),nn.MaxPool2d(2),
          nn.Conv2d(32,64,3,padding=1,bias=False),nn.BatchNorm2d(64),nn.ReLU(True),nn.MaxPool2d(2),
          nn.Conv2d(64,96,3,padding=1,bias=False),nn.BatchNorm2d(96),nn.ReLU(True),nn.MaxPool2d(2),
          nn.Conv2d(96,128,3,padding=1,bias=False),nn.BatchNorm2d(128),nn.ReLU(True),nn.AdaptiveAvgPool2d((1,1)))
        self.projection=nn.Sequential(nn.Flatten(),nn.Linear(128,128),nn.ReLU(True),nn.Dropout(.10))
    def forward(self,x):return self.projection(self.features(x))
class Attn(nn.Module):
    def __init__(self):
        super().__init__(); self.attention=nn.Sequential(nn.Linear(512,128),nn.Tanh(),nn.Linear(128,1))
    def forward(self,x):
        w=torch.softmax(self.attention(x).squeeze(-1),1); return (x*w.unsqueeze(-1)).sum(1),w
class Encoder(nn.Module):
    def __init__(self):
        super().__init__(); self.frame_encoder=FrameCNN()
        self.temporal_encoder=nn.GRU(128,256,2,batch_first=True,bidirectional=True,dropout=.20)
        self.temporal_attention=Attn()
    def forward(self,x):
        B,T,C,H,W=x.shape; f=self.frame_encoder(x.reshape(B*T,C,H,W)).reshape(B,T,-1)
        h,_=self.temporal_encoder(f); return self.temporal_attention(h)[0]
class Model(nn.Module):
    def __init__(self):
        super().__init__(); self.encoder=Encoder()
        self.classifier=nn.Sequential(nn.LayerNorm(512),nn.Dropout(.2),nn.Linear(512,256),nn.ReLU(True),nn.Dropout(.2),nn.Linear(256,44))
    def forward(self,x):return self.classifier(self.encoder(x))
model=Model().to(DEVICE)
ssl=torch.load(SSL_PATH,map_location="cpu",weights_only=False)
model.encoder.load_state_dict(ssl["encoder_state_dict"],strict=True)
print("SSL best epoch:",ssl.get("best_epoch"),"| val loss:",ssl.get("best_val_ssl_loss"))
print("Params:",sum(p.numel() for p in model.parameters()))

In [ ]:
# CELL 4 — Loss / optimizer / metrics
cnt=tr.model_class_id.value_counts().reindex(range(44),fill_value=0).values
w=np.clip(len(tr)/(44*np.maximum(cnt,1)),.25,4).astype(np.float32)
criterion=nn.CrossEntropyLoss(weight=torch.tensor(w,device=DEVICE))
opt=torch.optim.AdamW([{"params":model.encoder.parameters(),"lr":1e-4},{"params":model.classifier.parameters(),"lr":3e-4}],weight_decay=1e-4)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",factor=.5,patience=3,min_lr=1e-6)
scaler=torch.amp.GradScaler("cuda",enabled=AMP) if DEVICE.type=="cuda" else None
def metric(y,p):
    pred=p.argmax(1); acc=(pred==y).mean(); top=np.argsort(-p,1)[:,:3]
    top3=np.mean([y[i] in top[i] for i in range(len(y))]); fs=[]
    for c in range(44):
        tp=np.sum((pred==c)&(y==c)); fp=np.sum((pred==c)&(y!=c)); fn=np.sum((pred!=c)&(y==c))
        pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
        fs.append(2*pr*rc/(pr+rc) if pr+rc else 0)
    return {"accuracy":float(acc),"macro_f1":float(np.mean(fs)),"top3":float(top3)},pred
def run(loader,training):
    model.train(training); L=N=0; Y=[];P=[]
    for x,y in tqdm(loader,leave=False):
        x=x.to(DEVICE);y=y.to(DEVICE)
        if training:opt.zero_grad(set_to_none=True)
        ctx=torch.enable_grad() if training else torch.inference_mode()
        with ctx,torch.amp.autocast(DEVICE.type,enabled=AMP):
            o=model(x);loss=criterion(o,y)
        if training:
            scaler.scale(loss).backward();scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(model.parameters(),5)
            scaler.step(opt);scaler.update()
        q=torch.softmax(o.detach().float(),1);Y.extend(y.cpu().numpy());P.append(q.cpu().numpy())
        L+=float(loss)*len(y);N+=len(y)
    Y=np.asarray(Y);P=np.concatenate(P);m,p=metric(Y,P);m["loss"]=L/N
    return m,Y,P,p
print("Training setup ready.")

In [ ]:
# CELL 5 — Fine-tune using TRAIN/VAL only
history=[];bestf=-1;bestep=0;bad=0
for ep in range(1,EPOCHS+1):
    t=time.time();tm,*_=run(train_loader,True);vm,*_=run(val_loader,False);sched.step(vm["macro_f1"])
    if vm["macro_f1"]>bestf+1e-6:
        bestf=vm["macro_f1"];bestep=ep;bad=0
        torch.save({"epoch":ep,"model_state_dict":model.state_dict(),"val_metrics":vm},BEST)
    else:bad+=1
    history.append({"epoch":ep,**{"train_"+k:v for k,v in tm.items()},**{"val_"+k:v for k,v in vm.items()}})
    pd.DataFrame(history).to_csv(HISTORY,index=False,encoding="utf-8-sig")
    print(f"Epoch {ep:03d} | train acc {tm['accuracy']:.4f} f1 {tm['macro_f1']:.4f} | val acc {vm['accuracy']:.4f} f1 {vm['macro_f1']:.4f} top3 {vm['top3']:.4f} | best {bestf:.4f}@{bestep} | {time.time()-t:.1f}s")
    if bad>=PATIENCE: print("Early stopping.");break
print("Best epoch:",bestep,"Best val F1:",bestf)

## Final test gate
Cell ต่อไปเปิด test 259 clips **ครั้งเดียว** หลังเลือก best model แล้ว อย่าปรับ hyperparameters จากผล test ชุดนี้

In [ ]:
# CELL 6 — FINAL TEST + baseline comparison
best=torch.load(BEST,map_location=DEVICE,weights_only=False);model.load_state_dict(best["model_state_dict"],strict=True);model.eval()
testm,Y,P,pred=run(test_loader,False)
print("="*80);print("FINAL TEST")
print(f"Accuracy : {testm['accuracy']:.4f} ({100*testm['accuracy']:.2f}%)")
print(f"Macro F1 : {testm['macro_f1']:.4f}")
print(f"Top-3    : {testm['top3']:.4f} ({100*testm['top3']:.2f}%)")
base={"accuracy":.2625,"macro_f1":.2327,"top3":.4826}
print("\\nFrozen Experiment 0 comparison")
for k in base:print(k,":",f"{base[k]:.4f} -> {testm[k]:.4f}  delta {testm[k]-base[k]:+.4f}")

In [ ]:
# CELL 7 — Export deployment checkpoint
deploy={"format":"VocaLipThai-44Word-v1","model_state_dict":model.state_dict(),
"model_config":{"classes":44,"frames":16,"roi":112,"feature_dim":128,"gru_hidden":256,"gru_layers":2},
"preprocessing":{"channel_order":"cached order; no channel swap","mean":MEAN.tolist(),"std":STD.tolist()},
"id_to_word":id2word,"word_to_id":word2id,"test_metrics":testm,
"scope":"in-domain closed vocabulary; NOT speaker-disjoint"}
torch.save(deploy,DEPLOY)
rec={"ssl_best_epoch":ssl.get("best_epoch"),"finetune_best_epoch":best["epoch"],"val":best["val_metrics"],
"test":testm,"baseline":{"accuracy":.2625,"macro_f1":.2327,"top3":.4826},"deploy":str(DEPLOY)}
RESULT.write_text(json.dumps(rec,ensure_ascii=False,indent=2),encoding="utf-8")
print("="*80);print("DEPLOYMENT CHECKPOINT READY")
print(DEPLOY)
print(f"Test Acc {100*testm['accuracy']:.2f}% | F1 {testm['macro_f1']:.4f} | Top3 {100*testm['top3']:.2f}%")
print("Next: real-time webcam/video inference")